In [ ]:
# Fix metadata

In [41]:
from methyltrain.config.loader import load_config
from methyltrain.fs.layout import ProjectLayout

from methyltrain.utils.utils import extract_batch_id
from methyltrain.utils.load_utils import load_audit_table, save_audit_table, save_metadata

from methyltrain.pipeline.download import build_metadata, build_biospecimen
from methyltrain.pipeline.audit import update_metadata

import anndata as ad

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [50]:
layout = ProjectLayout(
    project_name = "TCGA-UVM",
    root_dir = "../data",
    raw_dir = "../data/raw/TCGA-UVM",
    audit_table = "../data/metadata/TCGA-UVM/TCGA-UVM_audit_table.csv",
    metadata = "../data/metadata/TCGA-UVM/TCGA-UVM_metadata.csv",
    manifest = "../data/metadata/TCGA-UVM/TCGA-UVM_manifest.csv",
    status_log = "../data/metadata/TCGA-UVM/TCGA-UVM_status_log.csv",
    project_adata = "../data/processed/TCGA-UVM_adata.h5ad"
)
layout.initialize()
layout.validate()

config = load_config("../config/TCGA-UVM_config.yaml")


adata = ad.read_h5ad(layout.project_adata)
audit_table = load_audit_table(layout)

metadata = build_metadata(audit_table, config, verbose = False)
biospecimen = build_biospecimen(metadata, config, verbose = False)
metadata = metadata.reset_index()
metadata = metadata.merge(biospecimen[['aliquot_id', 'barcode']], 
                          on = 'aliquot_id', how = 'left')
metadata = metadata.set_index('file_id', drop = True)

metadata['batch_id'] = metadata['barcode'].apply(extract_batch_id)

# Update the audit table with the metadata
audit_table = update_metadata(audit_table, metadata)

# Clean the metadata of verbose output to the standard format
metadata = metadata.loc[metadata['status'] == 'success']

save_metadata(metadata, layout)
save_audit_table(audit_table, layout)

# Create a mapping from file_name → file_id using the index
fileid_map = metadata.index.to_series()
fileid_map.index = metadata['file_name']  # use file_name as the Series index

# Map file_id into adata.obs
adata.obs['file_id'] = adata.obs['file_name'].map(fileid_map)

# Set as index
adata.obs = adata.obs.set_index('file_id', drop=True)

adata.write_h5ad(layout.project_adata)